In [2]:
import schedule
import time
from datetime import datetime, timedelta
import mysql.connector
import requests


# MySQL 연결 설정
def create_connection():
    """MySQL 연결을 생성하고 반환."""
    return mysql.connector.connect(
        host="127.0.0.1",  # 또는 적절한 호스트 주소
        port=3306,
        user="root",
        password="tiger",
        database="ocean",
        pool_name="mypool",
        pool_size=5
    )


# MySQL 연결 객체
conn = None


# MySQL 연결 상태 확인 및 재연결
def ensure_connection():
    global conn
    if conn is None or not conn.is_connected():
        print("Database connection is not active. Attempting to reconnect...")
        conn = create_connection()
        print("Reconnected to the database.")


# API에서 데이터 가져오기
def fetch_api_data():
    url = "http://marineweather.nmpnt.go.kr:8001/openWeatherNow.do"
    params = {
        "serviceKey": "2B93BD36-A99E-413E-9582-F6428745D972",
        "resultType": "json",
        "mmaf": "101",
        "mmsi": "994401597",
        "dataType": "2"
    }

    # API 요청
    response = requests.get(url, params=params)

    # 응답 확인
    if response.status_code == 200:
        try:
            return response.json()  # JSON 데이터를 반환
        except Exception as e:
            print(f"Error parsing JSON response: {e}")
            return None
    else:
        print(f"Failed to fetch API data. Status code: {response.status_code}")
        return None



# 데이터 삽입 또는 업데이트
def upsert_data(data):
    try:
        ensure_connection()  # 연결 확인 및 재연결
        insert_query = """
        INSERT INTO oceandata (
            datetime, mmaf_code, mmaf_name, mmsi_code, mmsi_name, wind_direct, wind_speed,
            surface_curr_drc, surface_curr_speed, air_temperature, humidity, air_pressure,
            water_temperature, salinity, latitude, longitude
        )
        VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
        ON DUPLICATE KEY UPDATE
            wind_direct = VALUES(wind_direct),
            wind_speed = VALUES(wind_speed),
            surface_curr_drc = VALUES(surface_curr_drc),
            surface_curr_speed = VALUES(surface_curr_speed),
            air_temperature = VALUES(air_temperature),
            humidity = VALUES(humidity),
            air_pressure = VALUES(air_pressure),
            water_temperature = VALUES(water_temperature),
            salinity = VALUES(salinity);
        """

        with conn.cursor() as cursor:
            if "result" in data and "recordset" in data["result"]:
                for entry in data["result"]["recordset"]:
                    cursor.execute(insert_query, (
                        datetime.strptime(entry["DATETIME"], "%Y%m%d%H%M%S"),
                        entry["MMAF_CODE"],
                        entry["MMAF_NM"],
                        entry["MMSI_CODE"],
                        entry["MMSI_NM"],
                        float(entry["WIND_DIRECT"]) if entry["WIND_DIRECT"] != "미제공" else None,
                        float(entry["WIND_SPEED"]) if entry["WIND_SPEED"] != "미제공" else None,
                        float(entry["SURFACE_CURR_DRC"]) if entry["SURFACE_CURR_DRC"] != "미제공" else None,
                        float(entry["SURFACE_CURR_SPEED"]) if entry["SURFACE_CURR_SPEED"] != "미제공" else None,
                        float(entry["AIR_TEMPERATURE"]) if entry["AIR_TEMPERATURE"] != "미제공" else None,
                        float(entry["HUMIDITY"]) if entry["HUMIDITY"] != "미제공" else None,
                        float(entry["AIR_PRESSURE"]) if entry["AIR_PRESSURE"] != "미제공" else None,
                        float(entry["WATER_TEMPER"]) if entry["WATER_TEMPER"] != "미제공" else None,
                        float(entry["SALINITY"]) if entry["SALINITY"] != "미제공" else None,
                        round(float(entry["LATITUDE"]), 5),
                        round(float(entry["LONGITUDE"]), 5)
                    ))
            conn.commit()
        print("Data successfully inserted or updated.")
    except mysql.connector.Error as e:
        print(f"MySQL error: {e}")
    except Exception as e:
        print(f"Unexpected error: {e}")


# 작업 스케줄링
def scheduled_task():
    print(f"Running scheduled task at {datetime.now()}")
    api_data = fetch_api_data()
    upsert_data(api_data)


# 실행
if __name__ == "__main__":
    # 초기 연결 설정
    ensure_connection()

    # 초기 데이터 삽입
    print("Fetching initial data...")
    api_data = fetch_api_data()
    upsert_data(api_data)

    # 스케줄러 설정
    schedule.every(10).minutes.do(scheduled_task)

    print("Scheduler is running... Press Ctrl+C to stop.")
    try:
        while True:
            schedule.run_pending()
            time.sleep(1)
    except KeyboardInterrupt:
        print("Scheduler stopped. Closing database connection...")
        if conn.is_connected():
            conn.close()
            print("Database connection closed.")


Database connection is not active. Attempting to reconnect...
Reconnected to the database.
Fetching initial data...
Data successfully inserted or updated.
Scheduler is running... Press Ctrl+C to stop.
Running scheduled task at 2025-01-15 15:40:51.457070
Data successfully inserted or updated.
Scheduler stopped. Closing database connection...
Database connection closed.
